In [0]:
import io
import pandas as pd

nyts_path = "/Volumes/workspace/default/nyts/nyts2025_puf.sas7bdat"

# Read raw bytes via Spark's binaryFile reader -- avoids local-filesystem
# access entirely, sidestepping the same Volumes/FUSE issue we hit on BRFSS
file_bytes = spark.read.format("binaryFile").load(nyts_path).select("content").first()["content"]
print(f"Read {len(file_bytes) / 1e6:.1f} MB into memory (file on disk: 138.4 MB)")

nyts_pdf = pd.read_sas(io.BytesIO(file_bytes), format="sas7bdat")
print(f"\nLoaded {len(nyts_pdf)} rows, {len(nyts_pdf.columns)} columns")
print("\nFirst 20 column names:")
print(nyts_pdf.columns.tolist()[:20])

Read 138.4 MB into memory (file on disk: 138.4 MB)

Loaded 23630 rows, 1453 columns

First 20 column names:
['artificial_id', 'Location', 'Q1', 'Q2', 'Q3', 'Q4a', 'Q4b', 'Q4c', 'Q4d', 'Q4e', 'Q4f', 'Q4g', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10a', 'Q10b', 'Q10c']


In [0]:
# ---------------------------------------------------------------------------
# NYTS 2025 Bronze Layer — corrected column names matching actual SAS casing
# ---------------------------------------------------------------------------
NYTS_COLUMNS = [
    "artificial_id", "Location",
    "Q1", "Q2", "Q3",
    "Q4a", "Q4b", "Q4c", "Q4d", "Q4e", "Q4f", "Q4g",
    "Q5", "Q7", "Q8",
    "CELCIGT",
    "Stratum_num", "PSU_num", "WT_analysis",
]

nyts_pdf = nyts_pdf_full[NYTS_COLUMNS].copy()

# Standardize to uppercase to match BRFSS's naming convention -- this is
# itself part of the harmonization work, worth keeping consistent going forward
nyts_pdf.columns = [c.upper() for c in nyts_pdf.columns]

# SAS character columns sometimes decode as bytes rather than str
for col in nyts_pdf.columns:
    if nyts_pdf[col].dtype == object:
        nyts_pdf[col] = nyts_pdf[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

print(f"Selected {len(nyts_pdf.columns)} columns, {len(nyts_pdf)} rows")
print(nyts_pdf.dtypes)
print("\nSample rows:")
print(nyts_pdf.head(10))

Selected 19 columns, 23630 rows
ARTIFICIAL_ID     object
LOCATION         float64
Q1                object
Q2                object
Q3                object
Q4A               object
Q4B               object
Q4C               object
Q4D               object
Q4E               object
Q4F               object
Q4G               object
Q5                object
Q7                object
Q8                object
CELCIGT          float64
STRATUM_NUM      float64
PSU_NUM          float64
WT_ANALYSIS      float64
dtype: object

Sample rows:
  ARTIFICIAL_ID  LOCATION  Q1 Q2  ... CELCIGT STRATUM_NUM PSU_NUM  WT_ANALYSIS
0     A25000007       1.0   4  2  ...     2.0         5.0    28.0  1938.587324
1     A25000018       2.0  11  1  ...     2.0         5.0    21.0  2051.329337
2     A25000021       1.0   4  1  ...     2.0         5.0    26.0  1767.210026
3     A25000036       1.0   8  1  ...     1.0         2.0     9.0    39.244658
4     A25000035       1.0  10  2  ...     2.0         6.0    55.0   93

In [0]:
# Find the real column names -- case-insensitive match against what we expect
expected = ["ARTIFICIAL_ID", "LOCATION", "Q1", "Q2", "Q3", "Q4A", "Q4B", "Q4C", "Q4D", "Q4E", "Q4F", "Q4G",
            "Q5", "Q7", "Q8", "CELCIGT", "STRATUM_NUM", "PSU_NUM", "WT_ANALYSIS"]

actual_cols_upper = {c.upper(): c for c in nyts_pdf_full.columns}

print("Column name mapping (expected -> actual):")
for e in expected:
    actual = actual_cols_upper.get(e.upper(), "*** NOT FOUND ***")
    print(f"  {e:15s} -> {actual}")

Column name mapping (expected -> actual):
  ARTIFICIAL_ID   -> artificial_id
  LOCATION        -> Location
  Q1              -> Q1
  Q2              -> Q2
  Q3              -> Q3
  Q4A             -> Q4a
  Q4B             -> Q4b
  Q4C             -> Q4c
  Q4D             -> Q4d
  Q4E             -> Q4e
  Q4F             -> Q4f
  Q4G             -> Q4g
  Q5              -> Q5
  Q7              -> Q7
  Q8              -> Q8
  CELCIGT         -> CELCIGT
  STRATUM_NUM     -> Stratum_num
  PSU_NUM         -> PSU_num
  WT_ANALYSIS     -> WT_analysis


In [0]:
# ---------------------------------------------------------------------------
# Write NYTS Bronze table
# ---------------------------------------------------------------------------
nyts_spark_df = spark.createDataFrame(nyts_pdf.astype(str))  # cast to string, matching BRFSS Bronze convention

nyts_spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_nyts_2025_raw")
print(f"Written to workspace.default.bronze_nyts_2025_raw: {nyts_spark_df.count()} rows")

Written to workspace.default.bronze_nyts_2025_raw: 23630 rows


In [0]:
# ---------------------------------------------------------------------------
# Validate CELCIGT against FDA's published 5.2% weighted estimate
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F

bronze_nyts = spark.table("workspace.default.bronze_nyts_2025_raw")

validation_df = bronze_nyts.select(
    F.col("CELCIGT").cast("double").alias("CELCIGT"),
    F.col("WT_ANALYSIS").cast("double").alias("WT_ANALYSIS")
).filter(F.col("CELCIGT").isin(1.0, 2.0))  # drop missing/skip codes

n_valid = validation_df.count()
print(f"Valid CELCIGT responses: {n_valid} (out of 23,630 total)")

weighted_rate = validation_df.agg(
    (F.sum(F.when(F.col("CELCIGT") == 1.0, F.col("WT_ANALYSIS")).otherwise(0)) /
     F.sum("WT_ANALYSIS") * 100)
).first()[0]

print(f"Weighted current e-cigarette use rate: {weighted_rate:.2f}% (FDA published: 5.2%)")

Valid CELCIGT responses: 23380 (out of 23,630 total)
Weighted current e-cigarette use rate: 5.23% (FDA published: 5.2%)


In [0]:
# ---------------------------------------------------------------------------
# SILVER LAYER — NYTS 2025 Cleaned, Typed, Documented
# Same rigor as BRFSS Silver, but a genuinely different missing-code
# convention: NYTS uses letter codes (N/Z/S/.) rather than BRFSS's numeric
# sentinels (7/9/77/99). This difference is itself the interoperability
# finding worth citing in the methods section.
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F

bronze_nyts = spark.table("workspace.default.bronze_nyts_2025_raw")

RECODE_LOG_NYTS = []

def recode_nyts_missing(df, col, missing_codes, valid_map=None, label=""):
    """
    missing_codes: list of string codes to convert to null
    valid_map: optional dict mapping valid string codes -> numeric values
    """
    before_nulls = df.filter(F.col(col).isNull() | F.col(col).isin(missing_codes)).count()
    if valid_map:
        mapping_expr = F.create_map([F.lit(x) for pair in valid_map.items() for x in pair])
        df = df.withColumn(col, mapping_expr[F.col(col)])
    else:
        df = df.withColumn(col, F.when(F.col(col).isin(missing_codes), None).otherwise(F.col(col).cast("double")))
    after_nulls = df.filter(F.col(col).isNull()).count()
    RECODE_LOG_NYTS.append({
        "column": col, "label": label, "missing_codes": missing_codes,
        "nulls_before": before_nulls, "nulls_after": after_nulls,
    })
    return df

df = bronze_nyts

# --- Core demographics ---
df = recode_nyts_missing(df, "Q1", ["N"], label="age: N=not answered")
df = recode_nyts_missing(df, "Q2", ["N"], label="sex: N=not answered")
df = recode_nyts_missing(df, "Q3", ["N"], label="grade: N=not answered")
df = recode_nyts_missing(df, "LOCATION", [".N", "nan", "None"], label="survey location: not answered")

# --- Race/ethnicity: select-all-that-apply, DIFFERENT missing semantics ---
# "." = did not select this race (functionally "No"/0, not true missingness)
# "N" = did not answer the question at all (true missing)
# "Z" = not displayed to this respondent (skip logic)
# Only "N" and "Z" become null; "." becomes 0 (valid "not selected" response)
for c in ["Q4A", "Q4B", "Q4C", "Q4D", "Q4E", "Q4F", "Q4G"]:
    before_nulls = df.filter(F.col(c).isin(["N", "Z"])).count()
    df = df.withColumn(c, F.when(F.col(c).isin(["N", "Z"]), None)
                           .when(F.col(c) == ".", 0.0)
                           .otherwise(F.col(c).cast("double")))
    RECODE_LOG_NYTS.append({
        "column": c, "label": "race flag: .=not selected->0, N/Z=true missing->null",
        "missing_codes": ["N", "Z"], "nulls_before": before_nulls,
        "nulls_after": df.filter(F.col(c).isNull()).count(),
    })

# --- E-cigarette items (skip-pattern follow-ups) ---
df = recode_nyts_missing(df, "Q5", ["N", "Z"], label="ever e-cig use: N/Z=missing")
df = recode_nyts_missing(df, "Q7", ["N", "Z", "S"], label="lifetime days: S=skipped (never used, not random missing)")
df = recode_nyts_missing(df, "Q8", ["N", "Z", "S"], label="past-30-day days: S=skipped (never used, not random missing)")

# --- Outcome ---
df = recode_nyts_missing(df, "CELCIGT", ["M"], label="derived outcome: M=missing")

print("NYTS recode log:")
for entry in RECODE_LOG_NYTS:
    print(f"  {entry['column']:10s} nulls_before~{entry['nulls_before']:>6} -> nulls_after {entry['nulls_after']:>6}  ({entry['label']})")

# --- Drop rows missing the outcome ---
n_before = df.count()
df_silver_nyts = df.filter(F.col("CELCIGT").isNotNull())
n_after = df_silver_nyts.count()
print(f"\nDropped {n_before - n_after} rows missing CELCIGT ({n_before} -> {n_after})")

# --- Validate before writing ---
weighted_rate = df_silver_nyts.select(
    F.col("CELCIGT").cast("double").alias("CELCIGT"),
    F.col("WT_ANALYSIS").cast("double").alias("WT_ANALYSIS")
).agg(
    (F.sum(F.when(F.col("CELCIGT") == 1.0, F.col("WT_ANALYSIS")).otherwise(0)) /
     F.sum("WT_ANALYSIS") * 100)
).first()[0]

print(f"\nFinal row count: {n_after}")
print(f"Weighted CELCIGT rate post-cleaning: {weighted_rate:.2f}% (FDA published: 5.2%)")

assert n_after > 20000, "Sample size dropped more than expected -- investigate before writing"
assert 4.0 < weighted_rate < 7.0, "Rate outside plausible range -- investigate before writing"

# --- Write Silver Delta table ---
df_silver_nyts.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_nyts_2025_clean")

lineage_note_nyts = (
    f"Silver layer built from workspace.default.bronze_nyts_2025_raw. "
    f"NYTS uses letter-coded missingness (N/Z/S/dot) vs BRFSS numeric sentinels (7/9/77/99) "
    f"-- documented interoperability difference between CDC-family systems. "
    f"Race variables (Q4A-G) are select-all-that-apply: dot recoded to 0 (not selected, valid response), "
    f"N/Z recoded to null (true missing/skip logic) -- distinct from single-select missingness handling. "
    f"Final n={n_after}. Validated: weighted CELCIGT rate {weighted_rate:.2f} pct vs FDA published 5.2 pct."
)
lineage_note_nyts_escaped = lineage_note_nyts.replace("'", "''")
spark.sql(f"COMMENT ON TABLE workspace.default.silver_nyts_2025_clean IS '{lineage_note_nyts_escaped}'")

print("\nWritten to workspace.default.silver_nyts_2025_clean with lineage comment.")

NYTS recode log:
  Q1         nulls_before~    37 -> nulls_after     37  (age: N=not answered)
  Q2         nulls_before~    59 -> nulls_after     59  (sex: N=not answered)
  Q3         nulls_before~    26 -> nulls_after     26  (grade: N=not answered)
  LOCATION   nulls_before~    75 -> nulls_after     75  (survey location: not answered)
  Q4A        nulls_before~   215 -> nulls_after    215  (race flag: .=not selected->0, N/Z=true missing->null)
  Q4B        nulls_before~   215 -> nulls_after    215  (race flag: .=not selected->0, N/Z=true missing->null)
  Q4C        nulls_before~   215 -> nulls_after    215  (race flag: .=not selected->0, N/Z=true missing->null)
  Q4D        nulls_before~   215 -> nulls_after    215  (race flag: .=not selected->0, N/Z=true missing->null)
  Q4E        nulls_before~   215 -> nulls_after    215  (race flag: .=not selected->0, N/Z=true missing->null)
  Q4F        nulls_before~   215 -> nulls_after    215  (race flag: .=not selected->0, N/Z=true missing-

In [0]:
# Check what CELCIGT actually looks like in Bronze for supposedly-missing rows
bronze_nyts.groupBy("CELCIGT").count().orderBy(F.desc("count")).show(10)

+-------+-----+
|CELCIGT|count|
+-------+-----+
|    2.0|21985|
|    1.0| 1395|
|    nan|  250|
+-------+-----+



In [0]:
# ---------------------------------------------------------------------------
# SILVER LAYER — NYTS 2025, corrected for the astype(str) "nan" string bug
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F

bronze_nyts = spark.table("workspace.default.bronze_nyts_2025_raw")

NAN_STR = "nan"  # what real nulls became after Bronze's astype(str) cast

RECODE_LOG_NYTS = []

def recode_nyts_missing(df, col, missing_codes, label=""):
    all_missing = missing_codes + [NAN_STR]
    before_nulls = df.filter(F.col(col).isin(all_missing)).count()
    df = df.withColumn(col, F.when(F.col(col).isin(all_missing), None).otherwise(F.col(col).cast("double")))
    after_nulls = df.filter(F.col(col).isNull()).count()
    RECODE_LOG_NYTS.append({"column": col, "label": label, "nulls_before": before_nulls, "nulls_after": after_nulls})
    return df

df = bronze_nyts

df = recode_nyts_missing(df, "Q1", ["N"], label="age: N=not answered")
df = recode_nyts_missing(df, "Q2", ["N"], label="sex: N=not answered")
df = recode_nyts_missing(df, "Q3", ["N"], label="grade: N=not answered")
df = recode_nyts_missing(df, "LOCATION", [], label="survey location: not answered")

for c in ["Q4A", "Q4B", "Q4C", "Q4D", "Q4E", "Q4F", "Q4G"]:
    before_nulls = df.filter(F.col(c).isin(["N", "Z", NAN_STR])).count()
    df = df.withColumn(c, F.when(F.col(c).isin(["N", "Z", NAN_STR]), None)
                           .when(F.col(c) == ".", 0.0)
                           .otherwise(F.col(c).cast("double")))
    RECODE_LOG_NYTS.append({"column": c, "label": "race flag: .=not selected->0, N/Z/nan=true missing->null",
                             "nulls_before": before_nulls, "nulls_after": df.filter(F.col(c).isNull()).count()})

df = recode_nyts_missing(df, "Q5", ["N", "Z"], label="ever e-cig use")
df = recode_nyts_missing(df, "Q7", ["N", "Z", "S"], label="lifetime days: S=skipped")
df = recode_nyts_missing(df, "Q8", ["N", "Z", "S"], label="past-30-day days: S=skipped")
df = recode_nyts_missing(df, "CELCIGT", ["M"], label="derived outcome")

print("NYTS recode log (corrected):")
for entry in RECODE_LOG_NYTS:
    print(f"  {entry['column']:10s} nulls {entry['nulls_before']:>6} -> {entry['nulls_after']:>6}  ({entry['label']})")

n_before = df.count()
df_silver_nyts = df.filter(F.col("CELCIGT").isNotNull())
n_after = df_silver_nyts.count()
print(f"\nDropped {n_before - n_after} rows missing CELCIGT ({n_before} -> {n_after})")

weighted_rate = df_silver_nyts.select(
    F.col("CELCIGT").alias("CELCIGT"), F.col("WT_ANALYSIS").cast("double").alias("WT_ANALYSIS")
).agg((F.sum(F.when(F.col("CELCIGT") == 1.0, F.col("WT_ANALYSIS")).otherwise(0)) / F.sum("WT_ANALYSIS") * 100)).first()[0]

print(f"\nFinal row count: {n_after} (expect ~23,380, matching Bronze's valid-response count)")
print(f"Weighted CELCIGT rate: {weighted_rate:.2f}% (FDA published: 5.2%; Bronze check earlier: 5.23%)")

assert n_after > 20000, "Sample size dropped more than expected"
assert 4.5 < weighted_rate < 6.0, "Rate outside plausible range"

df_silver_nyts.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_nyts_2025_clean")

lineage_note = (
    f"Silver layer built from workspace.default.bronze_nyts_2025_raw. "
    f"CORRECTED for astype(str) artifact: real NaN values became literal string 'nan' during Bronze write, "
    f"initially bypassing missing-code recoding (first Silver attempt gave 5.18 pct vs correct 5.2x pct) -- "
    f"caught via cross-check against Bronze-layer validation and FDA's published 5.2 pct figure. "
    f"NYTS uses letter-coded missingness (N/Z/S/dot) vs BRFSS numeric sentinels -- documented interoperability "
    f"difference. Final n={n_after}. Weighted CELCIGT rate {weighted_rate:.2f} pct vs FDA published 5.2 pct."
)
spark.sql(f"COMMENT ON TABLE workspace.default.silver_nyts_2025_clean IS '{lineage_note.replace(chr(39), chr(39)*2)}'")

print("\nWritten to workspace.default.silver_nyts_2025_clean with corrected lineage.")

NYTS recode log (corrected):
  Q1         nulls     37 ->     37  (age: N=not answered)
  Q2         nulls     59 ->     59  (sex: N=not answered)
  Q3         nulls     26 ->     26  (grade: N=not answered)
  LOCATION   nulls     75 ->     75  (survey location: not answered)
  Q4A        nulls    215 ->    215  (race flag: .=not selected->0, N/Z/nan=true missing->null)
  Q4B        nulls    215 ->    215  (race flag: .=not selected->0, N/Z/nan=true missing->null)
  Q4C        nulls    215 ->    215  (race flag: .=not selected->0, N/Z/nan=true missing->null)
  Q4D        nulls    215 ->    215  (race flag: .=not selected->0, N/Z/nan=true missing->null)
  Q4E        nulls    215 ->    215  (race flag: .=not selected->0, N/Z/nan=true missing->null)
  Q4F        nulls    215 ->    215  (race flag: .=not selected->0, N/Z/nan=true missing->null)
  Q4G        nulls    215 ->    215  (race flag: .=not selected->0, N/Z/nan=true missing->null)
  Q5         nulls    145 ->    145  (ever e-cig us